In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"      # hide TensorFlow's start-up messages
from pathlib import Path
import numpy as np
import pandas as pd
import keras
import tensorflow as tf

keras.utils.set_random_seed(42)
tf.config.experimental.enable_op_determinism()
DATA = Path("data")

## How often each pixel is used
Count, for every pixel, how many filter positions cover it.

In [ ]:
def usage(n, f, pad=0, stride=1):
    """How many windows cover each pixel of an n x n image (padding cells not counted)."""
    m = n + 2 * pad
    count = np.zeros((m, m), int)
    for i in range(0, m - f + 1, stride):
        for j in range(0, m - f + 1, stride):
            count[i:i + f, j:j + f] += 1
    return count[pad:pad + n, pad:pad + n]

print("no padding:\n", usage(5, 3))
print("padding 1:\n", usage(5, 3, pad=1))
pd.DataFrame(usage(5, 3)).to_csv(DATA / "usage_valid.csv", index=False)
pd.DataFrame(usage(5, 3, pad=1)).to_csv(DATA / "usage_padded.csv", index=False)

## Zero padding by hand

In [ ]:
X = np.array([[3, 0, 1, 2, 7], [1, 5, 8, 9, 3], [2, 7, 2, 5, 1], [0, 1, 3, 1, 7], [4, 2, 1, 6, 2]], float)
Xp = np.pad(X, 1)                                  # one ring of zeros: 5 x 5 -> 7 x 7
print(Xp.astype(int))
K = np.array([[1, 0, -1], [1, 0, -1], [1, 0, -1]], float)
layer = keras.layers.Conv2D(1, 3, padding="same", use_bias=False)
layer.build((None, 5, 5, 1))
layer.set_weights([K.reshape(3, 3, 1, 1)])
print("Keras 'same' output:\n", layer(X.reshape(1, 5, 5, 1)).numpy()[0, :, :, 0].astype(int))

## valid against same: three convolution layers on MNIST-sized input

In [ ]:
def three_convs(padding, strides=1):
    return keras.Sequential([keras.Input(shape=(28, 28, 1))] +
                            [keras.layers.Conv2D(32, 3, padding=padding, strides=strides, activation="relu")
                             for _ in range(3)])

for padding, strides in (("valid", 1), ("same", 1), ("same", 2)):
    m = three_convs(padding, strides)
    print(padding, "strides", strides, ":", [l.output.shape[1:3] for l in m.layers])

## Strides: the worked examples

In [ ]:
def out_size(n, f, p=0, s=1):
    return (n + 2 * p - f) // s + 1

def tf_size(h, w, f, p=0, s=1):
    x = tf.zeros((1, h, w, 1))
    k = tf.zeros((f, f, 1, 1))
    return tuple(tf.nn.conv2d(x, k, strides=s, padding=[[0, 0], [p, p], [p, p], [0, 0]]).shape[1:3])

print("7x7, f=3, s=2:       ", out_size(7, 3, s=2), tf_size(7, 7, 3, s=2))
print("7x7, f=3, p=1, s=2:  ", out_size(7, 3, p=1, s=2), tf_size(7, 7, 3, p=1, s=2))
print("6 rows x 7 cols, s=2:", (out_size(6, 3, s=2), out_size(7, 3, s=2)), tf_size(6, 7, 3, s=2))
print("28, f=3, p=1, s=2:   ", out_size(28, 3, p=1, s=2), out_size(14, 3, p=1, s=2), out_size(7, 3, p=1, s=2))

## The formula against TensorFlow for many settings

In [ ]:
rows = []
for n in range(5, 33):
    for f in (1, 2, 3, 5, 7):
        for p in (0, 1, 2):
            for s in (1, 2, 3):
                if n + 2 * p >= f:
                    rows.append(dict(n=n, f=f, p=p, s=s, formula=out_size(n, f, p, s), tf=tf_size(n, n, f, p, s)[0]))
check = pd.DataFrame(rows)
print(len(check), "settings; formula equals TensorFlow in", (check.formula == check.tf).sum())

## Work done by one filter: stride 1 against stride 2
Each output value needs f x f multiplications.

In [ ]:
for s in (1, 2, 3):
    o = out_size(224, 3, p=1, s=s)
    print(f"224 x 224 image, 3x3 filter, padding 1, stride {s}: output {o} x {o}, "
          f"{o * o * 9:,} multiplications")